 # RWF-2000 Batch Processing

Procesuje všechna videa z `data/RWF-2000/RWF-2000/{train,val}/{Fight,NonFight}/` přes RT-DETR + ViTPose pipeline.

**Output:** Pickle soubory v `data/RWF-2000/skeletons/{train,val}/{Fight,NonFight}/<video_id>.pkl` se strukturou:
```python
{
    'video_path': str,
    'label': int,                    # 0=NonFight, 1=Fight
    'sampled_frame_indices': [int],  # original frame indices in video
    'person_tracks': [
        {
            'person_id': int,
            'frame_indices': [int],  # which sampled frames had this person
            'skeletons': np.ndarray, # (T, 14, 3) CrowdPose keypoints + score
            'bboxes': np.ndarray,    # (T, 5) [x1,y1,x2,y2,conf]
        },
        ...
    ]
}
```

**Time estimate:** ~16-20 hodin na RTX 3090 pro celý dataset (2000 videí, 5 FPS sampling).

In [1]:
# === Cell 0: Imports + setup ===
import os
import sys
import glob
import pickle
import time
import numpy as np
import cv2
import torch
from tqdm.auto import tqdm

# Insert project root for relative imports
sys.path.insert(0, os.getcwd())

# Reusenuje detektory z suspicious_detector.py (RTDETRDetector, VitPoseExtractor)
# - tyto třídy jsou identické s těmi v extract_skeletons_two_stage.ipynb
from suspicious_detector import (
    RTDETRDetector, VitPoseExtractor,
    _RTDETR_CONFIG, _RTDETR_MODEL,
    _VITPOSE_CONFIG, _VITPOSE_MODEL,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

C:\Users\tomas\anaconda3\envs\ocv412cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


C:\Users\tomas\anaconda3\envs\ocv412cuda\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# === Cell 1: Configuration ===
RWF_ROOT     = 'data/RWF-2000/RWF-2000'           # input videos
OUTPUT_ROOT  = 'data/RWF-2000/skeletons'           # output skelety

SAMPLE_FPS = 5            # extract 5 frames per second from each video
                          # RWF videos are ~5s @ 30fps = 150 frames; 5 FPS gives 25 frames per video
                          # 2000 videos × 25 frames = 50k frames total

# Tracking parameters (simple greedy centroid)
MAX_TRACK_DISTANCE = 100  # pixels for matching
MAX_MISSED_FRAMES = 3     # how many frames without detection before track ends
MIN_TRACK_LENGTH = 3      # discard tracks shorter than this

# RT-DETR settings
RTDETR_CONF = 0.5         # higher confidence for cleaner data (RWF doesn't need multi-pass)

In [3]:
# === Cell 2: Load models ===
print('Loading RT-DETR + ViTPose...')
rtdetr = RTDETRDetector(_RTDETR_CONFIG, _RTDETR_MODEL, device=DEVICE)
vitpose = VitPoseExtractor(_VITPOSE_CONFIG, _VITPOSE_MODEL, device=DEVICE)
print('✓ Models ready')

Loading RT-DETR + ViTPose...
  ✓ RT-DETR loaded on cuda
Loads checkpoint by local backend from path: models/vitpose/vitpose-h-multi-crowdpose.pth
  ✓ ViTPose loaded on cuda
✓ Models ready


In [4]:
# === Cell 3: Helper - extract frames at sampled FPS ===
def extract_video_frames(video_path, sample_fps=SAMPLE_FPS):
    """Read video, sample frames at sample_fps. Return (frames_list, frame_indices)."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return [], []
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, int(round(video_fps / sample_fps)))

    frames, indices = [], []
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            frames.append(frame)
            indices.append(idx)
        idx += 1
    cap.release()
    return frames, indices

In [5]:
# === Cell 4: Simple greedy centroid tracker ===
def get_centroid(skeleton_xy):
    """Centroid of valid keypoints (T, K, 2) -> (2,) or None."""
    if skeleton_xy is None or len(skeleton_xy) == 0:
        return None
    valid = ~np.all(skeleton_xy == 0, axis=-1) & ~np.isnan(skeleton_xy).any(axis=-1)
    if valid.sum() == 0:
        return None
    return skeleton_xy[valid].mean(axis=0)


def _split_tracks(tracks, max_missed):
    active, complete = [], []
    for t in tracks:
        (complete if t['missed'] >= max_missed else active).append(t)
    return active, complete


def track_persons(per_frame_skeletons, per_frame_bboxes,
                  max_dist=MAX_TRACK_DISTANCE, max_missed=MAX_MISSED_FRAMES,
                  min_length=MIN_TRACK_LENGTH):
    """
    Greedy centroid tracking přes framy.

    Args:
        per_frame_skeletons: list[np.ndarray (P_t, K, 3)] - per-frame ViTPose output
        per_frame_bboxes:    list[np.ndarray (P_t, 5)]    - RT-DETR boxes
        max_dist: max pixel distance for matching
        max_missed: kolik framů bez detekce před ukončením tracku
        min_length: minimální délka tracku

    Returns:
        list of person tracks (dict with skeletons, bboxes, frame_indices)
    """
    tracks = []      # active tracks
    completed = []   # finished tracks

    for fi, (skeletons, bboxes) in enumerate(zip(per_frame_skeletons, per_frame_bboxes)):
        if skeletons is None or len(skeletons) == 0:
            for t in tracks:
                t['missed'] += 1
            tracks, complete = _split_tracks(tracks, max_missed)
            completed.extend(complete)
            continue

        # Compute centroids of new detections
        centroids = [get_centroid(s[:, :2]) for s in skeletons]

        # Greedy match active tracks → nearest unmatched detection
        matched = set()
        for t in tracks:
            best_d, best_i = float('inf'), -1
            for i, c in enumerate(centroids):
                if i in matched or c is None:
                    continue
                d = float(np.linalg.norm(t['last'] - c))
                if d < best_d:
                    best_d, best_i = d, i
            if best_i >= 0 and best_d < max_dist:
                t['skeletons'].append(skeletons[best_i])
                t['bboxes'].append(bboxes[best_i])
                t['frames'].append(fi)
                t['last'] = centroids[best_i]
                t['missed'] = 0
                matched.add(best_i)
            else:
                t['missed'] += 1

        # Retire dead tracks
        tracks, complete = _split_tracks(tracks, max_missed)
        completed.extend(complete)

        # Start new tracks for unmatched detections
        for i, c in enumerate(centroids):
            if i not in matched and c is not None:
                tracks.append({
                    'skeletons': [skeletons[i]],
                    'bboxes': [bboxes[i]],
                    'frames': [fi],
                    'last': c,
                    'missed': 0,
                })

    completed.extend(tracks)  # flush remaining

    # Filter short tracks, convert to numpy
    result = []
    for i, t in enumerate(completed):
        if len(t['frames']) < min_length:
            continue
        result.append({
            'person_id': i,
            'frame_indices': t['frames'],
            'skeletons': np.array(t['skeletons']),
            'bboxes': np.array(t['bboxes']),
        })
    return result

In [6]:
# === Cell 5: Process single video ===
def process_video(video_path, label):
    """Process one video → dict with metadata + person tracks."""
    frames, frame_indices = extract_video_frames(video_path)
    if not frames:
        return None

    per_frame_skeletons = []
    per_frame_bboxes = []

    for frame in frames:
        # Stage 1: persons (single-pass detect, faster than better_detection)
        boxes = rtdetr.detect(frame, conf_threshold=RTDETR_CONF)

        if len(boxes) == 0:
            per_frame_skeletons.append(np.array([]))
            per_frame_bboxes.append(np.array([]))
            continue

        # Stage 2: skeletons via ViTPose
        skeletons = vitpose.extract_keypoints(frame, boxes)
        per_frame_skeletons.append(skeletons)
        per_frame_bboxes.append(boxes)

    tracks = track_persons(per_frame_skeletons, per_frame_bboxes)

    return {
        'video_path': video_path,
        'label': label,                          # 0=NonFight, 1=Fight
        'sampled_frame_indices': frame_indices,
        'n_persons_per_frame': [len(s) if s is not None and hasattr(s, '__len__') else 0
                                 for s in per_frame_skeletons],
        'person_tracks': tracks,
    }

In [7]:
# === Cell 6: Test on 3 videos (rychlý sanity check) ===
test_cases = [
    ('train/Fight', 1),
    ('train/NonFight', 0),
    ('val/Fight', 1),
]

t0 = time.time()
for split_dir, label in test_cases:
    in_dir = os.path.join(RWF_ROOT, split_dir)
    files = sorted([f for f in os.listdir(in_dir) if f.endswith('.avi')])
    if not files:
        print(f'⚠ No videos in {in_dir}')
        continue
    fp = os.path.join(in_dir, files[0])
    print(f'\n--- {fp} ---')
    result = process_video(fp, label)
    if result:
        print(f'  Sampled frames: {len(result["sampled_frame_indices"])}')
        print(f'  Person counts per frame: {result["n_persons_per_frame"]}')
        print(f'  Tracks: {len(result["person_tracks"])}')
        for t in result['person_tracks'][:3]:
            print(f'    Track {t["person_id"]}: T={len(t["frame_indices"])}, '
                  f'sk={t["skeletons"].shape}, bbox={t["bboxes"].shape}')
print(f'\nTotal test time: {time.time()-t0:.1f}s')


--- data/RWF-2000/RWF-2000\train/Fight\-1l5631l3fg_0.avi ---
  Sampled frames: 25
  Person counts per frame: [1, 1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
  Tracks: 2
    Track 0: T=25, sk=(25, 14, 3), bbox=(25, 5)
    Track 1: T=19, sk=(19, 14, 3), bbox=(19, 5)

--- data/RWF-2000/RWF-2000\train/NonFight\-1l5631l3fg_0.avi ---
  Sampled frames: 25
  Person counts per frame: [1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2]
  Tracks: 2
    Track 0: T=6, sk=(6, 14, 3), bbox=(6, 5)
    Track 1: T=16, sk=(16, 14, 3), bbox=(16, 5)

--- data/RWF-2000/RWF-2000\val/Fight\0Ow4cotKOuw_0.avi ---
  Sampled frames: 25
  Person counts per frame: [5, 4, 4, 3, 2, 2, 3, 4, 2, 4, 5, 4, 2, 4, 4, 5, 4, 6, 9, 8, 8, 9, 11, 7, 10]
  Tracks: 12
    Track 1: T=3, sk=(3, 14, 3), bbox=(3, 5)
    Track 3: T=5, sk=(5, 14, 3), bbox=(5, 5)
    Track 7: T=22, sk=(22, 14, 3), bbox=(22, 5)

Total test time: 10.9s


In [8]:
# === Cell 7: Batch processing function ===
def process_split(split_name, skip_existing=True):
    """Process všechna videa v daném splitu (train nebo val)."""
    for class_dir, label in [('Fight', 1), ('NonFight', 0)]:
        in_dir = os.path.join(RWF_ROOT, split_name, class_dir)
        out_dir = os.path.join(OUTPUT_ROOT, split_name, class_dir)
        os.makedirs(out_dir, exist_ok=True)

        videos = sorted([f for f in os.listdir(in_dir) if f.endswith('.avi')])
        print(f'\n=== {split_name}/{class_dir}: {len(videos)} videos ===')

        ok = 0
        fail = 0
        skipped = 0
        for vname in tqdm(videos, desc=f'{split_name}/{class_dir}'):
            out_path = os.path.join(out_dir, vname.replace('.avi', '.pkl'))
            if skip_existing and os.path.exists(out_path):
                skipped += 1
                continue

            try:
                result = process_video(os.path.join(in_dir, vname), label)
                if result is None:
                    fail += 1
                    continue
                with open(out_path, 'wb') as f:
                    pickle.dump(result, f)
                ok += 1
            except Exception as e:
                print(f'  ⚠ {vname}: {e}')
                fail += 1

        print(f'  ✓ {ok} success, ⊘ {skipped} skipped, ✗ {fail} failed')

In [9]:
# === Cell 8: SPUSTIT BATCH (uncomment níže) ===
# Pozor: 16-20 hodin na full RTX 3090 dataset (2000 videí × 25 framů × ~150 ms/frame)
# Můžeš nechat běžet přes noc.

# process_split('train')
# process_split('val')

print('Ready. Uncomment lines above and run cell to start batch processing.')

Ready. Uncomment lines above and run cell to start batch processing.


In [10]:
# === Cell 9: Verify output ===
def verify_outputs():
    print(f'{"split":<10} {"class":<10} {"#pkl":>6}  sample size')
    print('-' * 60)
    total = 0
    for split in ['train', 'val']:
        for cls in ['Fight', 'NonFight']:
            d = os.path.join(OUTPUT_ROOT, split, cls)
            if not os.path.exists(d):
                print(f'{split:<10} {cls:<10} {0:>6}  (dir not yet created)')
                continue
            files = [f for f in os.listdir(d) if f.endswith('.pkl')]
            total += len(files)
            sample_info = ''
            if files:
                with open(os.path.join(d, files[0]), 'rb') as f:
                    sample = pickle.load(f)
                sample_info = f'{len(sample["person_tracks"])} tracks, ' \
                              f'{len(sample["sampled_frame_indices"])} frames'
            print(f'{split:<10} {cls:<10} {len(files):>6}  {sample_info}')
    print(f'\nTotal pickles: {total} / 2000 expected')

verify_outputs()

split      class        #pkl  sample size
------------------------------------------------------------
train      Fight           0  (dir not yet created)
train      NonFight        0  (dir not yet created)
val        Fight           0  (dir not yet created)
val        NonFight        0  (dir not yet created)

Total pickles: 0 / 2000 expected
